# Telegram Channel History Backfill (Google Colab)

This notebook scans Telegram channel media history, parses captions, and stores records in MongoDB Atlas with duplicate prevention.


In [ ]:
# 1) Mount Google Drive for session and output persistence
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/telegram_backfill'
SESSION_DIR = f'{BASE_DIR}/session'
OUTPUT_DIR = f'{BASE_DIR}/output'

import os
os.makedirs(SESSION_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Session dir:', SESSION_DIR)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# 2) Install dependencies
!pip -q install pyrogram tgcrypto pymongo python-dotenv tqdm


In [ ]:
# 3) Imports and config helpers
import json
import os
import re
from datetime import datetime, timezone
from getpass import getpass

from pyrogram import Client
from pyrogram.enums import MessageMediaType
from pymongo import MongoClient, ASCENDING
from tqdm.auto import tqdm

def get_required_env(name: str) -> str:
    value = os.getenv(name)
    if value:
        return value
    value = getpass(f'Enter {name}: ').strip()
    if not value:
        raise ValueError(f'Missing required environment variable: {name}')
    os.environ[name] = value
    return value

def parse_caption(caption: str | None) -> dict:
    caption = caption or ''
    # Parses `key: value` and `#tag` patterns into structured metadata
    kv_pairs = dict(re.findall(r'(?mi)^([a-z0-9_\- ]{2,50}):\s*(.+)$', caption))
    tags = re.findall(r'(?<!\w)#([\w_]+)', caption)
    return {
        'key_values': {k.strip().lower().replace(' ', '_'): v.strip() for k, v in kv_pairs.items()},
        'tags': sorted(set([t.lower() for t in tags])),
    }

print('Helpers loaded.')


In [ ]:
# 4) Set secure environment variables
# Prefer Colab Secrets / runtime env vars.
# Required: TG_API_ID, TG_API_HASH, MONGODB_URI

TG_API_ID = int(get_required_env('TG_API_ID'))
TG_API_HASH = get_required_env('TG_API_HASH')
MONGODB_URI = get_required_env('MONGODB_URI')

MONGODB_DB = os.getenv('MONGODB_DB', 'telegram_backfill')
MONGODB_COLLECTION = os.getenv('MONGODB_COLLECTION', 'messages')
TG_SESSION_NAME = os.getenv('TG_SESSION_NAME', 'telegram_user_session')

print('Environment configured (secrets hidden).')


In [ ]:
# 5) Configure channels to backfill
# Examples: '@public_channel', 'https://t.me/channel_name', or numeric chat id
CHANNELS = [
    # '@example_channel',
]

if not CHANNELS:
    print('⚠️ Add at least one channel to CHANNELS before running backfill.')
else:
    print('Channels configured:', CHANNELS)


In [ ]:
# 6) Connect MongoDB and ensure duplicate-prevention index
mongo_client = MongoClient(MONGODB_URI)
collection = mongo_client[MONGODB_DB][MONGODB_COLLECTION]
collection.create_index([('channel', ASCENDING), ('message_id', ASCENDING)], unique=True, name='uniq_channel_message_id')
print('MongoDB connected and index ensured.')


In [ ]:
# 7) Backfill media history from channels

SESSION_PATH = os.path.join(SESSION_DIR, TG_SESSION_NAME)

media_types = {
    MessageMediaType.PHOTO: 'photo',
    MessageMediaType.VIDEO: 'video',
    MessageMediaType.DOCUMENT: 'document',
    MessageMediaType.AUDIO: 'audio',
    MessageMediaType.VOICE: 'voice',
    MessageMediaType.ANIMATION: 'animation',
    MessageMediaType.VIDEO_NOTE: 'video_note',
    MessageMediaType.STICKER: 'sticker',
}

inserted = 0
updated = 0
skipped_non_media = 0
export_rows = []

with Client(
    name=SESSION_PATH,
    api_id=TG_API_ID,
    api_hash=TG_API_HASH,
    workdir=SESSION_DIR,
    in_memory=False,
) as app:
    print('Pyrogram client started.')

    for channel in CHANNELS:
        print(f'\n--- Scanning: {channel} ---')

        for msg in tqdm(app.get_chat_history(channel), desc=f'History {channel}'):
            if not msg.media or msg.media not in media_types:
                skipped_non_media += 1
                continue

            media_type = media_types[msg.media]
            media_obj = getattr(msg, media_type, None)

            caption_meta = parse_caption(msg.caption)

            doc = {
                'channel': str(channel),
                'chat_id': msg.chat.id if msg.chat else None,
                'message_id': msg.id,
                'date': msg.date.astimezone(timezone.utc).isoformat() if msg.date else None,
                'media_type': media_type,
                'file_id': getattr(media_obj, 'file_id', None),
                'file_unique_id': getattr(media_obj, 'file_unique_id', None),
                'caption_raw': msg.caption or '',
                'caption_meta': caption_meta,
                'views': getattr(msg, 'views', None),
                'forwards': getattr(msg, 'forwards', None),
                'collected_at': datetime.now(timezone.utc).isoformat(),
            }

            result = collection.update_one(
                {'channel': doc['channel'], 'message_id': doc['message_id']},
                {'$set': doc},
                upsert=True,
            )

            if result.upserted_id is not None:
                inserted += 1
            elif result.modified_count > 0:
                updated += 1

            export_rows.append(doc)

print('\nBackfill complete!')
print('Inserted:', inserted)
print('Updated:', updated)
print('Skipped non-media:', skipped_non_media)
print('Total exported rows:', len(export_rows))


In [ ]:
# 8) Save optional JSON export to Google Drive
export_file = os.path.join(
    OUTPUT_DIR,
    f"backfill_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)
with open(export_file, 'w', encoding='utf-8') as f:
    json.dump(export_rows, f, ensure_ascii=False, indent=2)

print('Export saved:', export_file)
